In [ ]:
from scipy.interpolate import griddata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import csv
from matplotlib.lines import Line2D
from scipy.integrate import quad
from functools import reduce
import operator
Cww = 1
Cgg = 0
Cbb = -1
mz = 91.188
mw = 80.3692
cw = mw/mz
sw = np.sqrt(1 - cw**2) 
alpha = 1/128
alpha_s = 0.11 #mz
hbar_c = 1.973e-13  # GeV·cm

Caa = Cww + Cbb
Caz = cw**2 * Cww - sw**2 * Cbb
Czz = cw**4 * Cww + sw**4 * Cbb

In [ ]:
L1 = 25 #3025
L2 = 150 #ab-1 #7407

griddensity = 400

In [ ]:
ma = [10, 30, 50, 70, 85]
S_eff = [0.62, 0.768, 0.732, 0.581, 0.162]
B_eff = [0.035, 0.035, 0.035, 0.035, 0.035]
B_cs = 84.29*0.178*0.174

# 读取数据
pho_ALP = pd.read_csv('./alp_decay_widths.csv')

# 将数据类型转换为更小的浮动数类型以减少内存使用
pho_ALP_tautau = 0.178*0.174*pho_ALP['xs_tautau'].astype('float32')
pho_ALP_lambda = 1000 / pho_ALP['Lambda']

xs = pho_ALP_tautau
gstar = pho_ALP_lambda
MQ = pho_ALP['ma']
# 为信号截面插值，生成从7到79GeV每隔2GeV的数据点
signal_xs_interp = np.interp(pho_ALP['ma'], ma, S_eff)

In [ ]:
#######################
##光子衰变函数
#######################
def Gamma_a_to_gg(ma_grid, Lambda_grid, fermions):
    Ceff = C_gamma_gamma_eff(ma_grid, fermions)
    return 4 * np.pi * alpha**2 * ma_grid**3 / Lambda_grid**2 * np.real(Ceff * np.conj(Ceff))
def C_gamma_gamma_eff(ma_grid, fermions):
    total = np.zeros_like(ma_grid, dtype=complex)
    for f in fermions:
        term = (
            f["Nc"] * f["Qf"]**2 * cff * B1_tau(f["mf"], ma_grid) / (16 * np.pi**2)
        )
        total += term
    return Caa + total + 2 * Cww * alpha / (np.pi * sw**2) * B2_tau(mw, ma_grid)
def c_qq(Qq, T3q, mq, ma_array):
    ln_mu2_mq2 = np.log(mz**2 / mq**2)
    ln_mu2_mZ2 = np.log(mz**2 / mz**2)
    ln_mu2_mW2 = np.log(mz**2 / mw**2)
    g_vals = g_tau(mq, ma_array)
    term1 = -12 * Qq**2 * alpha**2 * Caa * (ln_mu2_mq2 + g_vals)
    term2 = -12 * alpha**2 / (sw**2 * cw**2) * Caz * Qq * (T3q - 2 * Qq * sw**2) * (ln_mu2_mZ2 + 1.5)
    term3 = -12 * alpha**2 / (sw**4 * cw**4) * Czz * (Qq**2 * sw**4 - T3q * sw**2 * Qq + 1/8) * (ln_mu2_mZ2 + 0.5)
    term4 = -3 * alpha**2 /(sw**4) * (ln_mu2_mW2 + 0.5) * Cww
    return term1 + term2 + term3 + term4

def g_tau(ml, ma_array):
    g_vals = np.empty_like(ma_array, dtype=complex)
    small_mask = ml**2 < ma_array**2
    large_mask = ml**2 > ma_array**2
    # 小极限
    log_term = np.log(ma_array[small_mask]**2 / ml**2) - 1j * np.pi
    g_vals[small_mask] = -1/6 * log_term**2 + 2/3
    # 大极限
    g_vals[large_mask] = 7/3
    # 相等情况
    g_vals[~(small_mask | large_mask)] =  5 + 4/3*(-0.5 + 1j * np.pi/2)
    return g_vals

def build_sm_fermions(ma_grid):
    fermions = []
    sm_data = [
        # name, Qf, mf, T3, Nc
        ("u_L",  2/3, 2.2e-3,  +0.5, 3), ("u_R",  2/3, 2.2e-3,  0.0,  3),
        ("d_L", -1/3, 4.7e-3,  -0.5, 3), ("d_R", -1/3, 4.7e-3,  0.0,  3),
        ("c_L",  2/3, 1.28,    +0.5, 3), ("c_R",  2/3, 1.28,    0.0,  3),
        ("s_L", -1/3, 0.096,   -0.5, 3), ("s_R", -1/3, 0.096,   0.0,  3),
        ("t_L",  2/3, 173,     +0.5, 3), ("t_R",  2/3, 173,     0.0,  3),
        ("b_L", -1/3, 4.18,    -0.5, 3), ("b_R", -1/3, 4.18,    0.0,  3),
        ("e_L", -1.0, 0.511e-3,-0.5, 1), ("e_R", -1.0, 0.511e-3, 0.0, 1),
        ("mu_L",-1.0, 0.1057,  -0.5, 1), ("mu_R",-1.0, 0.1057,   0.0, 1),
        ("tau_L",-1.0, 1.777,  -0.5, 1), ("tau_R",-1.0, 1.777,   0.0, 1),
    ]
    for name, Qf, mf, T3, Nc in sm_data:
        cff = c_qq(Qf, T3, mf, ma_array=ma_grid)  # <-- 支持2D
        fermions.append({"name": name, "Nc": Nc, "Qf": Qf, "mf": mf, "cff": cff})
    return fermions
def B1_tau(ml, ma_grid):
    tau = 4 * ml**2 / ma_grid**2
    return 1 - tau * f_tau(ml, ma_grid)**2

def B2_tau(ml, ma_grid):
    tau = 4 * ml**2 / ma_grid**2
    return 1 - (tau - 1) * f_tau(ml, ma_grid)**2

def f_tau(ml, ma_grid):
    tau = 4 * ml**2 / ma_grid**2
    result = np.empty_like(tau, dtype=complex)
    mask1 = tau >= 1
    mask2 = tau < 1
    result[mask1] = np.arcsin(1 / np.sqrt(tau[mask1]))
    result[mask2] = (np.pi/2 + 0.5j * np.log((1 + np.sqrt(1 - tau[mask2])) / (1 - np.sqrt(1 - tau[mask2]))))
    return result

#######################
##轻子衰变函数
#######################
def Gamma_a_to_ll(ma_grid, ml, cell_eff, Lambda_grid):
    Gamma = np.zeros_like(ma_grid)
    mask = ma_grid >= 2 * ml
    m = ma_grid[mask]
    L = Lambda_grid[mask]
    kin = np.sqrt(1 - 4 * ml**2 / m**2)
    pref = m * ml**2 / (8 * np.pi * L**2)
    Gamma[mask] = pref * np.abs(cell_eff[mask])**2 * kin
    return Gamma

def compute_lepton_decay_widths(ma_grid, Lambda_grid):
    results = {}
    lepton_data = {
        "e":  (0.511e-3, -0.5),
        "mu": (0.1057,   -0.5),
        "tau":(1.777,    -0.5),
    }
    for name, (ml, T3) in lepton_data.items():
        c_eff = c_qq(Qq=-1.0, T3q=T3, mq=ml, ma_array=ma_grid)  # ma_array must be 2D
        Gamma = Gamma_a_to_ll(ma_grid, ml=ml, cell_eff=c_eff, Lambda_grid=Lambda_grid)
        results[name] = Gamma
    return results

#######################
##轻夸克和重夸克衰变
#######################
def compute_light_quark_decay_widths(ma_grid, Lambda_grid):
    alpha_s_vals = alpha_s * np.ones_like(ma_grid)
    results = {
        "uu": np.zeros_like(ma_grid, dtype=complex),
        "dd": np.zeros_like(ma_grid, dtype=complex),
        "ss": np.zeros_like(ma_grid, dtype=complex)
    }
    quark_groups = {
        "uu": [("u_L", 2/3, 2.2e-3, +0.5), ("u_R", 2/3, 2.2e-3, 0.0)],
        "dd": [("d_L", -1/3, 4.7e-3, -0.5), ("d_R", -1/3, 4.7e-3, 0.0)],
        "ss": [("s_L", -1/3, 0.096, -0.5), ("s_R", -1/3, 0.096, 0.0)],
    }
    for label, states in quark_groups.items():
        mq = states[0][2]  # threshold
        sum_cqq = np.zeros_like(ma_grid, dtype=complex)
        for name, Qf, mf, T3 in states:
            sum_cqq += c_qq(Qf, T3, mf, ma_array=ma_grid)
        
        form_factor = 1 + (83 / 4) * alpha_s_vals / np.pi
        prefactor = 32 * np.pi * alpha_s_vals**2 * ma_grid**3 / Lambda_grid**2
        amplitude = Cgg + sum_cqq / (32 * np.pi**2)
        amplitude_sq = amplitude * np.conj(amplitude)
        
        mask = ma_grid >= 2 * mq
        result = np.zeros_like(ma_grid, dtype=complex)
        result[mask] = prefactor[mask] * form_factor[mask] * amplitude_sq[mask]
        results[label] = np.real(result)
    return results

def Gamma_a_to_QQ(ma_grid, mQ, cQQ_eff, Lambda_grid):
    Gamma = np.zeros_like(ma_grid)
    mask = ma_grid >= 2 * mQ
    m = ma_grid[mask]
    L = Lambda_grid[mask]
    kin = np.sqrt(1 - 4 * mQ**2 / m**2)
    pref = 3 * m * mQ**2 / (8 * np.pi * L**2)
    ce2 = np.abs(cQQ_eff[mask])**2
    Gamma[mask] = pref * ce2 * kin
    return Gamma

def compute_heavy_quark_decay_widths(ma_grid, Lambda_grid):
    results = {}
    quarks = {
        "cc": [("c_L", 2/3, 1.28, +0.5), ("c_R", 2/3, 1.28, 0.0)],
        "bb": [("b_L", -1/3, 4.18, -0.5), ("b_R", -1/3, 4.18, 0.0)],
    }
    for label, quark_states in quarks.items():
        Gamma_sum = np.zeros_like(ma_grid)
        for name, Qf, mf, T3 in quark_states:
            c_eff = c_qq(Qq=Qf, T3q=T3, mq=mf, ma_array=ma_grid)
            Gamma = Gamma_a_to_QQ(ma_grid, mQ=mf, cQQ_eff=c_eff, Lambda_grid=Lambda_grid)
            Gamma_sum += Gamma
        results[label] = Gamma_sum
    return results

In [ ]:
Gamma_gg = Gamma_a_to_gg(MQ, 1000/gstar, build_sm_fermions(MQ))
Gamma_ll = compute_lepton_decay_widths(MQ, 1000/gstar)
Gamma_lqlq = compute_light_quark_decay_widths(MQ, 1000/gstar)
Gamma_qq = compute_heavy_quark_decay_widths(MQ, 1000/gstar)

In [ ]:
Gamma_ll_total   = reduce(operator.add, Gamma_ll.values())
Gamma_lqlq_total = reduce(operator.add, Gamma_lqlq.values())
Gamma_qq_total   = reduce(operator.add, Gamma_qq.values())
Gamma_total = Gamma_gg + Gamma_ll_total + Gamma_lqlq_total + Gamma_qq_total

In [ ]:
Ea = (mz**2 + MQ**2) / (2 * mz)
La = hbar_c * Ea / (MQ * Gamma_total)

In [ ]:
# Decay probability functions
def P_decay(Lmin, Lmax, Ld):
    return np.exp(-Lmin / Ld) - np.exp(-Lmax / Ld)

P_prompt = P_decay(0, 1, La)
P_llp    = P_decay(1, 200, La)
P_calo   = P_decay(200, 300, La)
P_Mono   = P_decay(450, 9e+99, La)

In [ ]:
P_frac = (P_prompt) / (P_prompt + P_llp + P_calo + P_Mono)

In [ ]:
# 创建网格，减少网格密度以降低内存使用
mq_14_os = np.linspace(pho_ALP['ma'].min(), pho_ALP['ma'].max(), griddensity)
gstar_14_os = np.linspace(gstar.min(), gstar.max(), griddensity)

# 创建网格
MQ_grid_14_os, Gstar_grid_14_os = np.meshgrid(mq_14_os, gstar_14_os)

# 构造插值用的点和目标点
points_frac = np.vstack((MQ, gstar)).T
P_frac_grid = griddata(points_frac, P_frac, (MQ_grid_14_os, Gstar_grid_14_os), method='linear')

# 使用二维插值来计算网格上的 S 和 Z
points_os = MQ,gstar.values
values_os = xs.values

# 对每个点进行信号和背景效率的插值
signal_efficiency = np.interp(MQ_grid_14_os.flatten(), ma, S_eff)
background_efficiency = np.interp(MQ_grid_14_os.flatten(), ma, B_eff)

# 将效率的插值结果重塑为(100, 100)形状
signal_efficiency_grid = signal_efficiency.reshape(MQ_grid_14_os.shape)
background_efficiency_grid = background_efficiency.reshape(MQ_grid_14_os.shape)


# 重新计算S_grid和背景
S_grid_14_os_emu = griddata(points_os, values_os, (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * P_frac_grid * L1*1000 * signal_efficiency_grid * 1000  # 25fb
background_14_os_emu = griddata(points_os, tuple([B_cs]*np.shape(values_os)[0]), (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * L1*1000 * background_efficiency_grid * 1000  # 25fb

# 计算显著度Z = S / sqrt(S + B)
Z_grid_14_os_emu = S_grid_14_os_emu / np.sqrt(S_grid_14_os_emu + background_14_os_emu)

# 创建30fb情况
# 用30fb进行计算
S_grid_14_os_30_emu = griddata(points_os, values_os, (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * P_frac_grid * L2*1000 * signal_efficiency_grid * 1000  
background_14_os_30_emu = griddata(points_os, tuple([B_cs]*np.shape(values_os)[0]), (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * L2*1000 * background_efficiency_grid * 1000 


# 计算显著度Z = S / sqrt(S + B)
Z_grid_14_os_30_emu = S_grid_14_os_30_emu / np.sqrt(S_grid_14_os_30_emu + background_14_os_30_emu)

def sigma_Z(s, b):
    term1 = (s**2 + 4*b*s + 4*b**2)*s / (4 * (s + b)**3)
    term2 = (s**2 * b) / (4 * (s + b)**3)
    return np.sqrt(term1 + term2)

std_3_emu=sigma_Z(S_grid_14_os_emu,background_14_os_emu)
#Z_3_amu= Z_grid_14_os_amu / np.sqrt(std_3_amu)
z3emup = Z_grid_14_os_emu+np.sqrt(std_3_emu)
z3emum = Z_grid_14_os_emu-np.sqrt(std_3_emu)

std_30_emu=sigma_Z(S_grid_14_os_30_emu,background_14_os_30_emu)
#Z_30_amu= Z_grid_14_os_30_amu / np.sqrt(std_30_amu)
z30emup = Z_grid_14_os_30_emu+np.sqrt(std_30_emu)
z30emum = Z_grid_14_os_30_emu-np.sqrt(std_30_emu)

In [ ]:
# 绘制单张图包含两个等值线图
plt.figure(figsize=(8, 6))

# Signal yield
N_prompt_150 = xs * (P_prompt + P_llp + P_calo) * 150 * 10**(6)  # pb \to ab
N_prompt_25 = xs * (P_prompt + P_llp + P_calo) * 25 * 10**(6) # pb \to ab
# Plot settings
levels = {'Prompt_150':7407, 'Prompt_25':3025}
colors = ['black', 'green']
labels = [R'Prompt: 150 ab$^{-1}$', R'Prompt: 25 ab$^{-1}$']
handles = []

plt.figure(figsize=(8,6))

# Z_grid_14_os_30
contour1 = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, Z_grid_14_os_30_emu, levels=[2], colors='blue')
#plt.clabel(contour1, fmt={2: fmt2}, inline=True, fontsize=10)
contour_emu30p = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z30emup, levels=[2], colors='blue',linestyles='--')
contour_emu30m = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z30emum, levels=[2], colors='blue',linestyles='--')
upper_segments_30 = contour_emu30p.allsegs[0]
lower_segments_30 = contour_emu30m.allsegs[0]
for upper_segment, lower_segment in zip(upper_segments_30, lower_segments_30):
    upper_coords = np.array(upper_segment)
    lower_coords = np.array(lower_segment)
    full_coords = np.vstack([upper_coords, lower_coords[::-1]])
    plt.fill(full_coords[:, 0], full_coords[:, 1], color='lightblue', alpha=0.5)

# Z_grid_14_os
contour2 = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, Z_grid_14_os_emu, levels=[2], colors='red')
#plt.clabel(contour2, fmt={2: fmt1}, inline=True, fontsize=10,manual=[(70,2e-3)])
contour_emu3p = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z3emup, levels=[2], colors='red',linestyles='--')
contour_emu3m = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z3emum, levels=[2], colors='red',linestyles='--')
upper_segments_3 = contour_emu3p.allsegs[0]
lower_segments_3 = contour_emu3m.allsegs[0]
for upper_segment, lower_segment in zip(upper_segments_3, lower_segments_3):
    upper_coords = np.array(upper_segment)
    lower_coords = np.array(lower_segment)
    full_coords = np.vstack([upper_coords, lower_coords[::-1]])
    plt.fill(full_coords[:, 0], full_coords[:, 1], color='lightpink', alpha=0.5)

#plt.title(r'ALP$\rightarrow \tau\tau \rightarrow e \mu $')
plt.ylim(1e-2,1e2)
plt.xlabel(R"m$_{a}$ [GeV]", fontsize=20)
plt.ylabel(R"$1/\Lambda$ [TeV]$^{-1}$", fontsize=20)
plt.yscale('log')
plt.tick_params(axis='x', labelsize=16)  # x 轴刻度字体大小
plt.tick_params(axis='y', labelsize=16)  # y 轴刻度字体大小
plt.grid()
legend_elements = [
    Line2D([0], [0], color='r', lw=2, label=R'2$\sigma$ C.L. at 25 ab$^{-1}$'),
    Line2D([0], [0], color='b', lw=2, label=R'2$\sigma$ C.L. at 150 ab$^{-1}$ '),
    Line2D([0], [0], color='r', lw=2, linestyle='--', label=R'25 ab$^{-1}$ 1$\sigma$ Error boundary'),
    Line2D([0], [0], color='b', lw=2, linestyle='--', label=R'150 ab$^{-1}$ 1$\sigma$ Error boundary')
]
plt.text(0.3, 0.1, R'$a \to \tau^{+}(\tau^{+}\to \mu^{+} \bar{v}_{\tau} v_{\mu}) \tau^{-}(\tau^{-}\to e^{-} v_{\tau} \bar{v}_{e})$', ha='center', va='center', transform=plt.gca().transAxes, fontsize=14, color='black')
plt.legend(handles=legend_elements, loc='lower right', fontsize=10)
plt.savefig('./atata.png',dpi=300)
plt.show()